In [ ]:
%config InteractiveShell.cache_size = 0
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np

import lucent
import matplotlib.pyplot as plt
from lucent.optvis import render, param, transform, objectives
from lucent.modelzoo import inceptionv1
from pathlib import Path
import torch
from lucent.optvis.objectives import wrap_objective, handle_batch
from torch.nn import functional as F
import numpy as np
from olt.tfms import transform, inverse_transform
from PIL import Image
from olt.act import InputOutputModelSnapshot
import torch
from olt.html_report import apply_cmap, to_pil, rd_bk_gn
from olt.shards import read_image_shard
import warnings
from tqdm import tqdm
from olt.feature_viz import tensor_to_img_array
from olt.feature_viz import get_feature_viz_input
import pandas as pd
from olt.tfms import transform, inverse_transform
from olt.act import InputOutputModelSnapshot, get_layer_activations
from olt.html_report import make_overlay_heatmap


device = "cpu"
model = inceptionv1(pretrained=True)
model = model.to(device)
model = model.eval()


layer_name = "mixed4e_1x1_pre_relu_conv"
channel = 55

plt.style.use("dark_background")

In [ ]:
def _receptive_block(i, ksize, stride, padding, input_size=None):
    """
    Returns [start, end) input indices (end=exclusive) that influence
    output position i of a conv layer.
    """
    start = i * stride - padding
    end = start + ksize

    if input_size is not None:
        start = max(start, 0)
        end = min(end, input_size)

    return start, end


@wrap_objective()
def neg_channel(layer, n_channel, batch=None):
    """Visualize a single channel"""
    @handle_batch(batch)
    def inner(model):
        res = model(layer)[:, n_channel].mean()
        # print(res)
        return res
    return inner


@wrap_objective()
def mse_at_position(layer, n_channel, pos, value, batch=None):
    # todo: remove this, the fn below is more useful
    y, x = pos
    @handle_batch(batch)
    def inner(model):
        o = model(layer)[:, n_channel]
        cur_val = o[:, y, x]
        # print("cur", cur_val, "us", value)
        res = F.mse_loss(o[:, y, x], value)
        return res
    return inner

@wrap_objective()
def mse_at_multiple_positions(layer, n_channel, positions, values, batch=None): 
    @handle_batch(batch)
    def inner(model):
        o = model(layer)
        mse = 0.
        for pos, val in zip(positions, values):
            y, x = pos
            mse += F.mse_loss(o[:, n_channel, y, x], val)
        return mse
    return inner


@wrap_objective()
def exact_tensor(layer, n_channel, value, batch=None):
    @handle_batch(batch)
    def inner(model):
        o = model(layer)[:, n_channel]
        res = F.mse_loss(o, value)
        return res
    return inner


@wrap_objective()
def exact_tensor_all_chans(layer, value, batch=None):
    @handle_batch(batch)
    def inner(model):
        o = model(layer)
        res = F.mse_loss(o, value)
        return res
    return inner


def get_batch_from_feature_vis(viz):
    img = np.floor(viz * 256).astype(np.uint8)
    timg = transform(Image.fromarray(img))[None]
    return timg



def make_heatmap(model, layer_name, neuron_selector, pil_img):
    # pil_img = Image.open(FLAT_IMAGES_BASE / f"{input_key}.jpeg")
    # idx = indices[sorted_idxs[0]]
    timg = transform(pil_img)[None]
    inv_img = inverse_transform(timg)
    ndl = NeuronDeepLift(model, model.get_submodule(layer_name))
    attr_res = ndl.attribute(timg, neuron_selector)

    size = (224, 224)

    overlay = attr_res[0].detach().cpu().sum(dim=0).numpy()
    overlay = overlay / np.abs(overlay).max()

    base = to_pil(inv_img[0], size=size)
    heat = apply_cmap(
        overlay,
        rd_bk_gn,
        vmin=-1,
        vmax=1,
        size=size,
        interpolation=Image.BILINEAR,
    )
    cell = Image.blend(base, heat, alpha=0.8)
    return cell

@wrap_objective()
def patch_across_channel(layer, patches, positions, batch=None):
    @handle_batch(batch)
    def inner(model):
        o = model(layer)
        # print("orig out shape", o.shape)
        mse = 0
        for patch, pos in zip(patches, positions):
            y, x = pos
            mse += F.mse_loss(o[:, :, y, x], patch)
        return mse
    return inner



@objectives.wrap_objective()
def patch_across_channel_with_ksize(layer, patches, positions, ksize, batch=None):
    @objectives.handle_batch(batch)
    def inner(model):
        o = model(layer)
        mse = 0
        for patch, pos in zip(patches, positions):
            y, x = pos
            y1 = y + ksize[0]
            x1 = x + ksize[1]
        
            mse += F.mse_loss(o[:, :, y:y1, x:x1].reshape(-1), patch)
        return mse
    return inner


@objectives.wrap_objective()
def pw_across_channel_with_ksize(layer, next_neuron_weight, pws, positions, ksize, stride, padding, batch=None):
    @objectives.handle_batch(batch)
    def inner(model):
        o = model(layer)
        mse = 0
        for pw, pos in zip(pws, positions):
            # y, x = pos
            y, y1 = _receptive_block(pos[0], ksize[0], stride[0], padding[0])
            x, x1 = _receptive_block(pos[1], ksize[1], stride[1], padding[1])
            pw0 = o[:, :, y:y1, x:x1].reshape(-1)*next_neuron_weight
            mse += F.mse_loss(pw0, pw)
        return mse
    return inner


In [ ]:
report_csv = Path("this-and-prev/mass-train-reports/mixed4e_1x1_pre_relu_conv/55/report.csv")
df = pd.read_csv(report_csv)
df.head()

In [ ]:
FLAT_BASE = Path("this-and-prev/flat-images")


In [ ]:
rendered_images = []
prev_layer_name = "mixed4d"
ksize = model.get_submodule(layer_name).kernel_size
next_neuron_layer = "mixed4e_1x1_pre_relu_conv"
next_neuron_channel = 55

fdf = df[df.cluster_label == 41].sample(n=4)
for row in fdf.itertuples():
    impath = FLAT_BASE / f"{row.input_image_key}.jpeg"
    batch = transform(Image.open(impath))[None]
    acts = InputOutputModelSnapshot.get_activations(batch, model, [prev_layer_name])
    patch = acts[prev_layer_name]["output"][0, :, row.y_position, row.x_position]
    next_neuron_weight = model.get_submodule(next_neuron_layer).weight[next_neuron_channel, :, :, :].detach().reshape(-1)
    with torch.no_grad():
        pw = (patch * next_neuron_weight)
    
    svizs = render.render_vis(
        model, 
        pw_across_channel_with_ksize(
            prev_layer_name, 
            next_neuron_weight,
            [pw], 
            [[row.y_position, row.x_position]],
            ksize,
        ),
        # patch_across_channel_with_ksize(prev_layer_name, [patch], [[row.y_position, row.x_position]], ksize),
        verbose=False, 
        show_image=False, 
        thresholds=(256,),
    )
    rendered_images.append(svizs[-1][0])

In [ ]:
_, axes = plt.subplots(1, 4)

for ax, im in zip(axes, rendered_images):
    ax.imshow(im)
plt.show()

In [ ]:
_, axes = plt.subplots(1, 4)

for ax, im in zip(axes, rendered_images):
    ax.imshow(im)
plt.show()

In [ ]:
df = pd.read_csv("this-and-prev/mass-train-reports/mixed4d_3x3_pre_relu_conv/31/report.csv")

In [ ]:
df.cluster_label.unique()

In [ ]:

render_viz_for_one_cluster(
    model, df, "mixed4d_3x3_pre_relu_conv", 31, "mixed4d_3x3_bottleneck", 21, (1,1), Path("hello"), "pw"
)

In [ ]:
report_csv = Path("this-and-prev/mass-train-reports/mixed4e_1x1_pre_relu_conv/55/report.csv")
df = pd.read_csv(report_csv)
df.head()


In [ ]:
from olt.show import show_single_channel_red_green_black as S
render_viz_for_one_cluster(
    df, "mixed4e_1x1_pre_relu_conv", 55, "mixed4d", 41, (0,0), Path("hello"), "pw"
)

In [ ]:

render_viz_for_one_cluster(
    df, "mixed4e_1x1_pre_relu_conv", 55, "mixed4d", 61, (0,0), Path("hello"), "pw"
)

In [ ]:
def render_viz_for_one_cluster(
    model,
    df,
    neuron_layer_name,
    neuron_channel,
    prev_layer_name,
    cluster_label,
    padding,
    dest,
    flat_images_base,
    opt_func="patch",
):
    rendered_images = []
    inv_images = []
    next_neuron_weight = (
        model.get_submodule(neuron_layer_name)
        .weight[neuron_channel, :, :, :]
        .detach()
        .reshape(-1)
    )
    layer = model.get_submodule(neuron_layer_name)
    ksize = layer.kernel_size
    # padding is passed because inception blocks have separate F.pad before each layer, which needs to be manually passed
    # padding = layer.padding
    stride = layer.stride

    fdf = df[df.cluster_label == cluster_label].sample(n=4)
    for row in fdf.itertuples():
        impath = flat_images_base / f"{row.input_image_key}.jpeg"
        batch = transform(Image.open(impath))[None]
        inv_image = inverse_transform(batch)

        acts = InputOutputModelSnapshot.get_activations(batch, model, [prev_layer_name])

        y, y1 = _receptive_block(row.y_position, ksize[0], stride[0], padding[0])
        x, x1 = _receptive_block(row.x_position, ksize[1], stride[1], padding[1])

        patch = acts[prev_layer_name]["output"][0, :, y:y1, x:x1].reshape(-1)

        if opt_func == "patch":
            obj = patch_across_channel_with_ksize(
                prev_layer_name,
                [patch],
                [[row.y_position, row.x_position]],
                ksize,
                ksize,
                stride,
                padding,
            )
        else:
            pw = patch * next_neuron_weight
            obj = pw_across_channel_with_ksize(
                prev_layer_name,
                next_neuron_weight,
                [pw],
                [[row.y_position, row.x_position]],
                ksize,
                stride,
                padding,
            )
        svizs = render.render_vis(
            model,
            obj,
            verbose=False,
            show_image=False,
            thresholds=(256,),
        )
        rendered_images.append(svizs[-1][0])
        inv_images.append(inv_image[0].permute(1, 2, 0).numpy())

    to_show = list(itertools.chain.from_iterable(zip(inv_images, rendered_images)))

    fig, axes = plt.subplots(1, len(to_show), figsize=(2 * len(to_show), 2))
    for ax, im in zip(axes, to_show):
        ax.imshow(im)
        ax.set_axis_off()
    plt.tight_layout()
    fig.savefig(dest / f"{cluster_label}.jpeg")
    plt.close()

    return to_show



In [ ]:
dest = Path("feature-viz-pw-4e55")
dest.mkdir(parents=True, exist_ok=True)


In [ ]:
_ = render_viz_for_one_cluster(61, dest, "pw")

In [ ]:
cluster_labels = df.cluster_label.unique()
cluster_labels

In [ ]:
# skip -1
cluster_labels = cluster_labels[1:]

In [ ]:
cid_by_name = {
    41: "Human Finger/Skin/Hand",
    58: "Background",
    53: "Dog Legs",
    24: "Letters",
    59: "Car",
    45: "Dog Stomach",
    31: "Human Face",
    43: "Food",
    50: "Mountain",
    48: "Human Leg Clothes",
    60: "Fox",
    61: "Cat",
    40: "Mushroom",
    32: "Human Faces (again)",
    14: "Corn-like",
    29: "Snake",
    44: "Soup-like",
    33: "Fish",
    23: "Thin rod-like",
    18: "Salamander",
    46: "White dog stomach",
    5: "criss-crossing white rods",
    26: "arc-like edge",
    16: "repeated vertical lines",
}
cluster_labels = list(cid_by_name.keys())
cluster_labels

In [ ]:
dest = Path("feature-viz-pw-4e55")
dest.mkdir(parents=True, exist_ok=True)
for i, cl in enumerate(cluster_labels):
    print(f"###### {i}/{len(cluster_labels)}")
    
    render_viz_for_one_cluster(
        model, df, "mixed4e_1x1_pre_relu_conv", 55, "mixed4d", cl, (0,0), Path("new-pws"), FLAT_BASE, "pw", 
    )
    # _ = render_viz_for_one_cluster(cl, dest, "pw")    
    
    

In [ ]:
to_show = render_viz_for_one_cluster(41, dest)

In [ ]:
fig, axes = plt.subplots(1, len(to_show), figsize=(2*len(to_show), 2))
for ax, im in zip(axes, to_show):
    ax.imshow(im)
    ax.set_axis_off()
plt.tight_layout()
fig.savefig("haha.jpeg")
plt.close()

# return to_show

In [ ]:
rendered_images = []
prev_layer_name = "mixed4d"
ksize = model.get_submodule(layer_name).kernel_size
# print(ksize)

fdf = df[df.cluster_label == 53].sample(n=4)
for row in fdf.itertuples():
    impath = FLAT_BASE / f"{row.input_image_key}.jpeg"
    batch = transform(Image.open(impath))[None]
    acts = InputOutputModelSnapshot.get_activations(batch, model, [prev_layer_name])
    patch = acts[prev_layer_name]["output"][0, :, row.y_position, row.x_position]
    
    svizs = render.render_vis(
        model, 
        patch_across_channel(prev_layer_name, [patch], [[row.y_position, row.x_position]]),
        verbose=False, 
        show_image=False, 
        thresholds=(256,),
    )
    rendered_images.append(svizs[-1][0])

In [ ]:
fig, axes = plt.subplots(1, 4)

for ax, im in zip(axes, rendered_images):
    ax.imshow(im)
plt.show()

# Collect for blog

In [ ]:
cid_by_name = {
    41: "Human Finger/Skin/Hand",
    58: "Background",
    53: "Dog Legs",
    24: "Letters",
    59: "Car",
    45: "Dog Stomach",
    31: "Human Face",
    43: "Food",
    50: "Mountain",
    48: "Human Leg Clothes",
    60: "Fox",
    61: "Cat",
    40: "Mushroom",
    32: "Human Faces (again)",
    14: "Corn-like",
    29: "Snake",
    44: "Soup-like",
    33: "Fish",
    23: "Thin rod-like",
    18: "Salamander",
    46: "White dog stomach",
    5: "criss-crossing white rods",
    26: "arc-like edge",
    16: "repeated vertical lines",
}

In [ ]:
import shutil
src = Path("feature-viz-pw-4e55")
blog_assets_dir = Path("feature-viz-of-pw-4e55")
blog_assets_dir.mkdir(exist_ok=True, parents=True)
def _get_fname(s):
    n = s.replace(" ", "-").replace("/", "-").replace("(", "-").replace(")", "-").lower()
    return f"{n}.jpeg"

for cid, name in cid_by_name.items():
    fn = _get_fname(name)
    shutil.copy(src / f"{cid}.jpeg", blog_assets_dir / _get_fname(name))
    

In [ ]:
! cp -R new-pws /Users/hariomnarang/Desktop/personal/blog/pages/posts/2026-06-19-disentangling-mixed4e-55/images/

In [ ]:
for cid, name in cid_by_name.items():
    fn = _get_fname(name)
    print(f"""## {name}\n\n![](./images/feature-viz-of-pw-4e55/{fn})\n""")

In [ ]:
print(f"""## {name}\n\n![](./images/feature-viz-of-patches-4e55/{fn})\n""")